In [10]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import average_precision_score
import torch

# Load and preprocess the data
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')
misconception_mapping = pd.read_csv('misconception_mapping.csv')

# Tokenization and feature extraction
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_data(data):
    return tokenizer(data['QuestionText'], data['AnswerText'], padding=True, truncation=True, return_tensors='pt')

# Fine-tune BERT model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=25)

# Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    evaluation_strategy="epoch",
    logging_dir='./logs',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    tokenizer=tokenizer,
    compute_metrics=lambda p: {'map@25': average_precision_score(p.predictions, p.label_ids)}
)

# Train the model
trainer.train()

# Make predictions on the test set
predictions = model.predict(test_data)

# Prepare submission file
submission = pd.DataFrame({
    'QuestionId_Answer': test_data['QuestionId_Answer'],
    'MisconceptionId': predictions
})

submission.to_csv('submission.csv', index=False)



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\RNCR4\anaconda3\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>={ACCELERATE_MIN_VERSION}'`